# BR05 — Five-arm evaluation and blinded review

Graph/evaluation environment with numpy, rouge-score and Pillow; no generator loading. Primary contrast is graph minus bidirectional retrieval. Includes pairwise-complete and all-arm-complete analyses, patient bootstrap intervals, completion-sensitive F1 and Holm correction across the selected generator family. Sparse/unknown labels remain masked. No ROUGE-L is computed for external diagnosis codes without independent report text.

BR04B users: this notebook now selects the **context-fit child config** by default. The first code cell prints its exact path. For the known validation run, this is `.../experiment02_v1/context_fit/v1/internal_val/config.json`. Changing `/vf/users` to `/data` alone does not fix a missing `context_fit/v1/internal_val` segment. Use BR07–BR09 for the new single-clinician workflow.


In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
# BR04B generated reports in a context-fit CHILD run, not the original BR00 run.
# Leave None to select context_fit/v1/<cohort>/config.json from the base config.
# Or set the exact CHILD config path printed by BR04B.
CONTEXT_CONFIG_PATH = None
BASE_CONFIG_PATH = EXP / 'config.json'
if not BASE_CONFIG_PATH.is_file():
    raise FileNotFoundError('Missing experiment02/config.json; select the project used for BR04B')
BASE_CFG = load_config(BASE_CONFIG_PATH)
CONFIG_PATH = (Path(CONTEXT_CONFIG_PATH).expanduser() if CONTEXT_CONFIG_PATH is not None
               else Path(BASE_CFG['run_dir']) / 'context_fit' / 'v1' / BASE_CFG['cohort'] / 'config.json')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        'BR04B child config not found: ' + str(CONFIG_PATH)
        + '\nSet CONTEXT_CONFIG_PATH to the exact child config printed by BR04B. '
        + 'Do not substitute the original run config: its generation paths and packet hashes differ.')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Evaluation config:', CONFIG_PATH)
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from generation import folder_for
FOLDERS = {key: folder_for(CFG, key, False, 1024 if key == 'qwen38' else 384)
           for key in ('pulse', 'llama_ecg', 'llama_base', 'qwen38')}
# To evaluate an intentionally smaller generator set, edit FOLDERS before the first evaluation.
# Optional exact completed folders may be assigned here BEFORE the preflight below.
# Keep each folder from this SAME context-fit run and non-thinking protocol.
TARGETS_PATH = (Path(CFG['shaoxing_cohort']) / 'targets.json' if CFG['cohort'] == 'shaoxing'
                else Path(CFG['frozen_out']) / 'targets.json')
MAPPING_REVIEW_NOTE = ''  # REQUIRED for Shaoxing: document completed clinical mapping/annotation review.
ALLOW_PARTIAL = False  # Explicit True for a labeled matched-complete-case analysis if failures remain.
MAX_RUN_HOURS = 30.0
EVALUATION_VERSION = 'v1'
DESTINATION = RUN / 'evaluation' / CFG['cohort'] / EVALUATION_VERSION
# Check paths before loading reference texts or starting the evaluation.
required = [RUN / 'protocol.json', RUN / 'packets' / CFG['cohort'] / 'manifest.json', TARGETS_PATH]
for model, folder in FOLDERS.items():
    print(model, '|', folder)
    required.extend([Path(folder) / 'protocol.json', Path(folder) / 'reports.jsonl'])
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Missing evaluation inputs:\n' + '\n'.join(missing)
        + '\nCheck CONTEXT_CONFIG_PATH, cohort, and each completed generator token budget. '
        + 'Do not copy protocol files between runs or regenerate reports just to repair this path.')
REFERENCE_TEXTS = {}
if CFG['cohort'] != 'shaoxing':
    from report_study import reference_texts
    from queries import load_queries
    parent_config = read(PROJECT / 'config.json')
    text_cases = [dict(q, query_id=q['id']) for q in load_queries(CFG)]
    REFERENCE_TEXTS = reference_texts(PROJECT, parent_config, text_cases)
# Optional independent external report texts may be supplied; never synthesize narratives from codes.
print('Evaluating:', {k:str(v) for k,v in FOLDERS.items()})


In [ ]:
# Diagnose cross-generator source differences without changing or bypassing evaluation.
# This reads the recorded provenance; it does NOT rerender shared query images.
from collections import Counter, defaultdict
import json
from common import atomic, sha
from evaluation import load_results
packet_path = RUN / 'packets' / CFG['cohort'] / 'manifest.json'
packet_manifest = read(packet_path)
packet_manifest['_file_sha256'] = sha(packet_path)
by_query = defaultdict(list)
rendering_environments = {}
for model, folder in FOLDERS.items():
    protocol = read(Path(folder) / 'protocol.json')
    rendering_environments[model] = {
        'libraries': protocol.get('libraries', {}),
        'helpers': protocol.get('helpers', {}),
    }
    for (query_id, arm), row in load_results(folder, packet_manifest).items():
        if row['status'] != 'ok':
            continue
        by_query[query_id].append({
            'model': model, 'arm': arm,
            'source_tensor_sha256': row.get('source_tensor_sha256'),
            'sensor_sha256': row.get('sensor_sha256'),
        })
affected = []
classification_counts = Counter()
for query_id, records in sorted(by_query.items()):
    tensors = {r['source_tensor_sha256'] for r in records}
    images = {r['sensor_sha256'] for r in records}
    missing = any(not r['source_tensor_sha256'] or not r['sensor_sha256'] for r in records)
    if missing:
        classification = 'missing_source_provenance'
    elif len(tensors) > 1:
        classification = 'waveform_tensor_mismatch'
    elif len(images) > 1:
        classification = 'same_waveform_different_png_file_hash'
    else:
        classification = 'consistent'
    classification_counts[classification] += 1
    if classification != 'consistent':
        grouped = defaultdict(list)
        for r in records:
            grouped[(r['source_tensor_sha256'], r['sensor_sha256'])].append(r['model'] + ':' + r['arm'])
        affected.append(dict(query_id=query_id, classification=classification,
            variants=[dict(source_tensor_sha256=tensor, sensor_sha256=image, model_arms=members)
                      for (tensor, image), members in grouped.items()]))
source_audit = dict(queries_checked=len(by_query), classifications=dict(classification_counts),
    affected_queries=affected, rendering_environments=rendering_environments,
    interpretation='Identical tensor hashes do not prove identical historical image pixels. '
        'A PNG hash difference may reflect metadata, compression, or actual rendering differences. '
        'Do not bypass the source-equality gate or rewrite report provenance.')
SOURCE_AUDIT_PATH = DESTINATION / 'input_source_audit.json'
atomic(SOURCE_AUDIT_PATH, source_audit)
print('Source classifications:', source_audit['classifications'])
for model, environment in rendering_environments.items():
    print(model, 'libraries:', environment['libraries'])
for case in affected[:3]:
    print(json.dumps(case, indent=2))
print('Full diagnostic:', SOURCE_AUDIT_PATH)


In [ ]:
from evaluation import evaluate
result = evaluate(CFG, FOLDERS, TARGETS_PATH, DESTINATION, reference_texts=REFERENCE_TEXTS,
                  allow_partial=ALLOW_PARTIAL, mapping_review_note=MAPPING_REVIEW_NOTE,
                  max_run_hours=MAX_RUN_HOURS, repeats=2000)
print('Complete:', result['complete'])
if result['complete']:
    print('Matched queries:', result['matched_queries'])
    for row in result['summary']: print(row)
print('Coverage:', DESTINATION / 'coverage.json')


In [ ]:
# LEGACY review packet. Use BR07 → BR08 → BR09 for the single-clinician workflow.
# This cell is disabled by default to avoid creating the older forms accidentally.
RUN_LEGACY_REVIEW = False
from review import prepare_review
N_REVIEW_QUERIES = 10  # Four models × five arms = 200 review items.
REVIEW_VERSION = 'v1'
if RUN_LEGACY_REVIEW:
    print(prepare_review(CFG, FOLDERS, RUN / 'review' / CFG['cohort'] / REVIEW_VERSION,
                         n_queries=N_REVIEW_QUERIES, seed=CFG['seed']))
else:
    print('Automated evaluation complete. Continue to BR07 for the new clinical review workflow.')
# Share ONLY PUBLIC. PRIVATE contains model/arm/query linkage.
